In [211]:
import numpy as np
from scipy import optimize
from scipy import stats

### generate hawkes synthetic data

In [103]:
# function for multivariate hawkes process
# code taken from https://arxiv.org/pdf/2502.18979 Algorithm 1

# mu - d dimensional vector
# alpha - dxd dimensional matrix
# beta - scalar > 0
# time - scalar > 0
# decay is alpha * beta * exp(-beta * (t-t_i)) - this is so we can limit 0<alpha<1, which is easier than 0<alpha<beta
def simulation_by_cluster_representation(mu, alpha, beta, time):
    d = len(mu) # dimension

    # family tree is not part of the paper's algorithm
    family_trees = [{} for _ in range(d)] # timestamp of event: (parent variable, timestamp of parent). parents of immigrants are -1

    # initialization
    T = [[] for _ in range(d)] # list of all immigrants and descendants
    A = [[] for _ in range(d)] # temporary list of ancestors

    # immigrant simulation
    for j in range(d):
        k = np.random.poisson(lam=mu[j]*time) # number of immigrants of type j

        # small_t = [[] for _ in range(d)] # t != T (from the paper's algorithm)
        small_t = []
        for _ in range(k):
            # small_t[j].append(np.random.uniform(low=0, high=time)) # small t is different from big T
            small_t.append(np.random.uniform(low=0, high=time))

        A[j] = small_t #small_t[j]
        T[j] = list(set(T[j] + A[j]))

        for immigrant in T[j]: # assigns the parent of each immigrant to -1
            family_trees[j][immigrant] = (j, -1)

    # helper function for while loop condition (self explanatory)
    def there_exists_at_least_one_j_st_Aj_neq_empty(A):
        for j in range(len(A)):
            if A[j] != []:
                return True
        return False

    # offspring simulation
    while there_exists_at_least_one_j_st_Aj_neq_empty(A):
        O = [[] for _ in range(d)] # offspring initialization
        for j in range(d):
            if A[j] != []:
                for l in range(len(A[j])):
                    for j_prime in range(d):
                        if alpha[j_prime][j] * beta > 0: # alpha * beta
                            k = np.random.poisson(lam=alpha[j_prime][j] * beta) # alpha * beta
                            
                            # finds elapsed time for descendants
                            # small_t = [[] for _ in range(d)]
                            small_t = []
                            for i in range(k):
                                # small_t[j_prime].append(np.random.exponential(beta))
                                small_t.append(np.random.exponential(beta))

                            # adds elapsed time of descendant to the timestamp of its parent (A[j][l] is the parent)
                            a_plus_t = []
                            for i in range(k):
                                a_plus_t.append(A[j][l] + small_t[i]) #small_t[j_prime][i])
                            O[j_prime] = list(set(O[j_prime] + a_plus_t))

                            # assigns the parent of each descendant
                            for descendant in a_plus_t:
                                family_trees[j_prime][descendant] = (j, A[j][l])

                        T[j_prime] = list(set(T[j_prime] + O[j_prime])) # offsprings are added to T
        
        A = O # offspring become ancestors

    # remove events beyond T and sort
    for j in range(d):
        i = 0
        while i < len(T[j]): # removes events that occur after time limit
            if T[j][i] > time:
                T[j].pop(i)
                i -= 1
            i += 1
        T[j] = sorted(T[j])

        # remove events from family_tree if they're beyond T
        for key in list(family_trees[j].keys()):
            if key > time:
                family_trees[j].pop(key)
    
    return T, family_trees

In [85]:
# generate data
# needs to make sure it generates enough points, otherwise estimation will be really off

# model parameters
mu = [0.4, 0.4] # background intensity
alpha = [[0.3, 0.4],
         [0.35, 0.45]] # mutual excitation matrix; alpha_12 means that 1 is excited by 2
beta = 1 # decay; assume beta is the same for all variables
time = 400 # time

np.random.seed(0)
timestamps, family_trees = simulation_by_cluster_representation(mu, alpha, beta, time) # hawkes

num_params = len(alpha) # used for flattening/reshaping alpha matrix

for i in timestamps:
    print(len(i), i)

# for tree in family_trees:
#     print()
#     for key in sorted(tree.keys()):
#         print(key, ':', tree[key])

622 [1.8781904770188262, 3.8239750564903012, 7.515920174542057, 7.67727932373341, 8.043018474997421, 8.087358976130288, 8.49710009720269, 8.59557842994215, 8.767555602284473, 8.895829291363096, 9.078638480807617, 10.562929612828103, 11.002031386468115, 11.588887086889027, 11.590851159013924, 11.831010488805859, 11.885313249658925, 12.721049575938688, 12.756343670866775, 12.840925555277416, 12.984427275397493, 13.12630038345756, 13.167126201985091, 13.274033604888556, 13.701142301434265, 13.737530555631327, 14.061383739437183, 14.69251722418571, 15.034767400902307, 15.67511690172827, 15.767852359114432, 15.801669054348295, 16.184578734775513, 16.70993985476551, 17.003764084256733, 17.24560935755364, 22.58640384757282, 24.090188651707933, 25.658998539513746, 25.70870982908992, 26.337942530386563, 27.666798182055217, 28.414423279154775, 29.80627573013195, 29.962558497587295, 31.05383098249805, 31.91863459448489, 34.85171988061629, 35.06408453326221, 37.00597504557781, 37.38434736594089, 3

### optimizing just the hawkes process

In [86]:
# function for log likelihood of multivariate hawkes process
# code taken from Laub's textbook "The Elements of Hawkes Processes" section 5.3 

# list of tuples (timestamp, variable that generated the timestamp)
def generate_timestamp_tuple(timestamps):
    timestamp_tuple = []
    for i in range(len(timestamps)):
        for time in timestamps[i]:
            timestamp_tuple.append((time, i))
    return sorted(timestamp_tuple, key=lambda x: x[0])

# from Laub's book
def calculate_intensity_matrix(alpha, timestamps, mu, beta, time):
    l = mu
    for t in timestamps:
        if t[0] < time:
            l = np.add(l, alpha[int(t[1].astype(np.int64))] * beta * np.exp(-beta * (time - t[0]))) # alpha[t[1]] is a 1D matrix
    return l

# from Laub's book
def calculate_compensator_matrix(alpha, timestamps, mu, beta, time):
    l = mu * time
    for t in timestamps:
        if t[0] < time:
            l = np.add(l, (1/beta) * alpha[int(t[1].astype(np.int64))] * beta * (1 - np.exp(-beta * (time - t[0]))))
    return l

# from Laub's book
def negative_log_likelihood(alpha, *args):
    timestamps, mu, beta, time, num_params = args
    timestamp_tuple = generate_timestamp_tuple(timestamps)
    alpha = alpha.reshape((num_params,num_params)) # alpha must be flattened and reshaped because optimize.fmin_l_bfgs_b requires a 1d array

    timestamp_tuple = np.array(timestamp_tuple)
    mu = np.array(mu)
    alpha = np.array(alpha)

    l = 0
    for t in timestamp_tuple:
        intensity = calculate_intensity_matrix(alpha, timestamp_tuple, mu, beta, t[0])
        l += np.log(intensity[int(t[1].astype(np.int64))])
        
    compensator = calculate_compensator_matrix(alpha, timestamp_tuple, mu, beta, time)
    for k in range(len(mu)):
        l -= compensator[k]
    return -l

In [87]:
# optimize multivariate hawkes parameters
# assume known mu and beta

alpha = np.array(alpha).flatten() # lbfgs requires flattening; will be unflattened later
args = (timestamps, mu, beta, time, num_params)

# for comparison
likelihood = negative_log_likelihood(alpha, *args)
print(likelihood)

initial_guess = [0.5 for _ in alpha] # arbitrary initialization
bounds = []
for i in range(len(alpha)):
    bounds.append((0.0001, 1-0.0001)) # bounds is (0,1), exclusive
bounds = np.array(bounds)

# do the optimization, with auto gradient
print('\noptimize with scipy-calculated gradient:')
opt_auto_grad = optimize.fmin_l_bfgs_b(func=negative_log_likelihood,
                                       args=args,
                                       x0=initial_guess,
                                       bounds=bounds,
                                       approx_grad=True)
for res in opt_auto_grad:
    print(res)

print('\nactual alpha:')
print(alpha)

alpha = alpha.reshape((num_params,num_params)) # reshape alpha for future use

457.579906933508

optimize with scipy-calculated gradient:
[0.33207295 0.42986741 0.33883438 0.42472836]
457.2531587750459
{'grad': array([ 0.00439968, -0.0016712 ,  0.00205773,  0.00131877]), 'task': 'CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH', 'funcalls': 50, 'nit': 8, 'warnflag': 0}

actual alpha:
[0.3  0.4  0.35 0.45]


### generating latent variables and parameters

In [224]:
# functions for generating latent variables

# Generate latent variables according to Gaussian distribution with specified variance
# z_j ~ N(0,sigma_z^2 I_d)  Z: shape = (p, d)
def generate_latent_Z(p, d, sigma_z):
    Z = np.random.normal(loc=0, scale=sigma_z, size=(p, d))
    return Z

# Generate Theta matrix with entries distributed according to Gaussian distribution with specified
# variance centered at value determined by pairwise distance between latent positions
# theta_jk = N(alpha - |z_j-z_k|^2,sigma_theta^2)   Theta: shape = (p, p)
def generate_theta(Z, alpha, sigma_theta):
    p = Z.shape[0]
    Theta = np.zeros((p, p))
    for j in range(p):
        for k in range(p):
            Theta[j, k] = np.random.normal(alpha - np.sum((Z[j] - Z[k])**2), sigma_theta)
    return Theta

def logistic(theta):
    theta_tilde = np.copy(theta)
    for i in range(len(theta)):
        for k in range(len(theta[i])):
            theta_tilde[i][k] = 1 / (1 + np.exp(-theta[i][k])) # np.log(1 + np.exp(theta[i][k]))

    return theta_tilde

In [248]:
# generating latent variables and theta

# np.random.seed(0)

p = len(mu) # number of latent nodes to generate
d = 2 # dimension of latent space
sigma_z = 1 # variance of latent space generation

alph = 1 # constant for theta generation; we call this alpha but i don't want to confuse it with the hawkes alpha
sigma_theta = 0.01 # variance for theta generation

Z = generate_latent_Z(p, d, sigma_z)
print("Z:")
print(Z)

theta = generate_theta(Z, alph, sigma_theta)
print("\ntheta:")
print(theta)

theta_tilde = logistic(theta)
print("\ntheta_tilde:")
print(theta_tilde)

for row in theta_tilde:
    if sum(row) >= 1:
        print("row sums to >= 1")

t, _ = simulation_by_cluster_representation(mu, theta_tilde, beta, time) # hawkes process using theta_tilde
print("\nt:")
print(t)

Z:
[[-0.29096432  1.22612848]
 [-0.305055   -0.43093152]]

theta:
[[ 1.00000399 -1.74499681]
 [-1.74327133  0.99811935]]

theta_tilde:
[[0.73105936 0.14867936]
 [0.14889789 0.73068866]]

t:
[[2.0645610170340056, 2.1354112661499873, 3.758392927786623, 5.072144486355361, 5.417790081168716, 5.772394220273646, 6.03476609545703, 6.618837880124479, 6.883452289693295, 10.380490075658733, 14.658039618675822, 14.685268112032647, 15.241064724061415, 17.247753435424936, 17.310460862151242, 17.781160603541483, 20.77431646862128, 21.670464003650327, 22.37509565009916, 23.740864388301564, 23.932843082517756, 24.218047776357636, 24.229926317469403, 24.43263749744218, 24.771904002174843, 24.919966314914255, 25.030854801411397, 25.306828839040136, 25.985699696007003, 26.43612980733974, 27.00403142147047, 27.203806707262107, 28.273914487877708, 28.46551989756231, 28.50008402879163, 28.840659101368445, 28.940129828574534, 29.08270373910556, 29.168668056259463, 29.316327531173414, 29.379921694113648, 29.3

i noticed that the theta_tilde values for the two variables are always very similar.
self excitations are always ~0.73 and mutual excitations are always ~0.05.
it makes sense that each variable is similar. the distance between the latent variables are the same so the mean/variance is the same.
it's interesting that theta_tilde always looks similar. this maybe is because 
also it doesn't matter what z or theta_tilde looks like, the optimization always estimates z to be [0,0,0,0].
in this sense it's not surprising that it's optimizing each variable to be the same. it's only surprising that self and mutual excitations are the same.

things to try:
- optimize z via just log_p_z
    - investigate why the likelihood is evaluating to greater than the actual
    - the pdf is just going to optimize to 0s because that's the mean
    - so it would make sense for log_p_z to optimize to 0s
    - in fact the thetas also optimize to the means (investigate this, but i suspect it's true)
- look at others' code
- check inputs
- do the optimized z values always sum to 0?

### optimizing the latent variables Z (doesn't work, but we're not doing this for now)

In [ ]:
# functions for negative complete data log-likelihood of the entire process

def euclidean_norm(z1, z2):
    total = 0
    for i in range(len(z1)):
        total += (z1[i] - z2[i]) ** 2
    return total
    # return np.sum((z1 - z2)**2)

# log(P(X | theta_tilde, theta))
def log_p_x_given_thetas(alpha, *args):
    # return -np.log(1 + np.exp(negative_log_likelihood(alpha, *args))) # uses the hawkes log likelihood
    return -negative_log_likelihood(alpha, *args) # hawkes log likelihood

# log(P(theta | Z))
def log_p_theta_given_z(sigma_theta, theta, alph, z, p):
    total = 0
    for j in range(p):
        for k in range(p): # range(p), not range(j,p)
            total -= np.log(np.sqrt(2 * np.pi * sigma_theta))
            
            numerator = (theta[j][k] - (alph - euclidean_norm(z[j], z[k]))) ** 2
            denominator = 2 * sigma_theta
            total -= numerator / denominator
    # print(total)
    return total

# log(P(Z))
def log_p_z(sigma_z, z, p, d):
    total = 0
    for j in range(p):
        for l in range(d):
            # total -= np.log(np.sqrt(2 * np.pi * sigma_z))

            # numerator = z[j][l] ** 2 # might be z[j][d-1]
            # denominator = 2 * sigma_z
            # total -= numerator / denominator
            total += stats.norm.logpdf(z[j][l], loc=0, scale=sigma_z)
    # print(total)
    return total

# log likelihood = log(P(X | theta_tilde, theta)) + log(P(theta | Z)) + log(P(Z))
def negative_complete_data_log_likelihood(z, *args):
    p, d, theta, alph, sigma_theta, sigma_z, theta_tilde, timestamps, mu, beta, time = args
    z = z.reshape(p,d) # z requres reshaping

    args = (timestamps, mu, beta, time, p) # for hawkes
    # print(log_p_x_given_thetas(theta_tilde, *args), log_p_theta_given_z(sigma_theta, theta, alph, z, p), log_p_z(sigma_z, z, p, d))
    return -(log_p_x_given_thetas(theta_tilde, *args) + log_p_theta_given_z(sigma_theta, theta, alph, z, p) + log_p_z(sigma_z, z, p, d))

In [249]:
# optimization for entire process (doesn't work)

theta_tilde = np.array(theta_tilde).flatten() # flatten theta_tilde for optimize.fmin_l_bfgs_b
Z = np.array(Z).flatten() # flatten z for optimize.fmin_l_bfgs_b
args = (p, d, theta, alph, sigma_theta, sigma_z, theta_tilde, t, mu, beta, time) # gathers variables

# for comparison
ll = negative_complete_data_log_likelihood(Z, *args)
print(ll)

# initial_guess = [0.0 for _ in Z] # arbitrary initial guess
initial_guess = []
for i in range(len(Z)):
    initial_guess.append(Z[i] - 0.001)

# do the optimization, with auto gradient
print('\noptimize with scipy-calculated gradient:')
opt_auto_grad = optimize.fmin_l_bfgs_b(func=negative_complete_data_log_likelihood,
                                       args=args,
                                       x0=initial_guess, # if initialize to actual, then the estimation is good, otherwise it's breaks
                                       approx_grad=True)
for res in opt_auto_grad:
    print(res)

print('\nactual Z:')
print(Z)

theta_tilde = theta_tilde.reshape((p,p)) # reshape theta_tilde
Z = Z.reshape(p,d) # reshape Z

-635.9269176809438

optimize with scipy-calculated gradient:
[ 0.00709355  0.8279829  -0.00699593 -0.82812325]
-636.1748119926635
{'grad': array([ 5.68434189e-05,  1.27329258e-03,  3.41060513e-05, -1.38697941e-03]), 'task': 'CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH', 'funcalls': 90, 'nit': 15, 'warnflag': 0}

actual Z:
[-0.29096432  1.22612848 -0.305055   -0.43093152]


### optimizing theta, not Z

In [243]:
# functions for optimizing (stopping before Z)

# log likelihood = log(P(X | theta_tilde, theta)) + log(P(theta | Z))
def negative_complete_data_log_likelihood_of_theta(theta, *args):
    p, d, z, alph, sigma_theta, theta_tilde, timestamps, mu, beta, time = args
    theta = theta.reshape(p,p) # theta requres reshaping

    args = (timestamps, mu, beta, time, p) # for hawkes
    return -(log_p_x_given_thetas(theta_tilde, *args) + log_p_theta_given_z(sigma_theta, theta, alph, z, p))

In [244]:
# optimization for entire process, up to theta

theta = np.array(theta).flatten() # flatten theta_tilde for optimize.fmin_l_bfgs_b
args = (p, d, Z, alph, sigma_theta, theta_tilde, t, mu, beta, time) # gathers variables

# for comparison
ll = negative_complete_data_log_likelihood_of_theta(theta, *args)
print(ll)

initial_guess = [0.5 for _ in theta] # arbitrary initial guess

# do the optimization, with auto gradient
print('\noptimize with scipy-calculated gradient:')
opt_auto_grad = optimize.fmin_l_bfgs_b(func=negative_complete_data_log_likelihood_of_theta,
                                       args=args,
                                       x0=initial_guess,
                                       approx_grad=True) # theta doesn't have bounds; theta_tilde does
for res in opt_auto_grad:
    print(res)

print('\nactual theta:')
print(theta)

theta = theta.reshape((p,p)) # reshape theta_tilde

344.78797030102515

optimize with scipy-calculated gradient:
[ 0.99999997 -8.71378573 -8.71378573  0.99999998]
344.7680176800916
{'grad': array([0., 0., 0., 0.]), 'task': 'CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL', 'funcalls': 25, 'nit': 3, 'warnflag': 0}

actual theta:
[ 1.01197211 -8.72819375 -8.70831595  1.00426746]


In [109]:
# error

# rmse
def rmse(actual, estimated):
    if len(actual) != len(estimated):
        actual = np.array(actual.flatten())

    err = 0
    for i in range(len(actual)):
        err += (actual[i] - estimated[i]) ** 2
    return np.sqrt(err)

In [111]:
print("rmse:")
print(rmse(theta, opt_auto_grad[0]))

rmse:
0.023169855985831954
